<a href="https://colab.research.google.com/github/Ahmed-Saeed-Abdullah-Alshanwany/Al_Game_Character_Designer/blob/main/Al_Game_Character_Designer_Gradio_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

🎮 Al Game Character Designer (Gradio Version)

Text-to-Image with Stable Diffusion + Interactive Ul

Optimized for Google Colab (T4 GPU)

Features:

Stable Diffusion 1.5

Memory optimized for T4

Interactive Gradio Ul

Control Prompt/Negative Prompt/Guidance/Steps / Seed

In [ ]:
# Install Required Libraries
!pip install -U diffusers transformers accelerate safetensors gradio

In [ ]:
import torch
import gradio as gr
from diffusers import StableDiffusionPipeline

Load Stable Diffusion Model

In [ ]:
model_id = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16
).to("cuda")

# Memory Optimization for Colab
pipe.enable_attention_slicing()
pipe.enable_vae_slicing()

print("Model loaded successfully!")

Define Generation Function

In [ ]:
def generate_character(prompt, negative_prompt, guidance, steps, seed):
    if int(seed) == 0:
        generator = None
    else:
        generator = torch.Generator("cuda").manual_seed(int(seed))

    image = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        guidance_scale=float(guidance),
        num_inference_steps=int(steps),
        generator=generator
    ).images[0]

    return image

Build Gradio Interface

In [ ]:
interface = gr.Interface(
    fn=generate_character,
    inputs=[
        gr.Textbox(label="Character Description", value="Cyberpunk warrior, neon armor, cinematic lighting"),
        gr.Textbox(label="Negative Prompt", value="blurry, low quality, distorted face"),
        gr.Slider(1, 15, value=7.5, label="Guidance Scale"),
        gr.Slider(10, 50, step=5, value=30, label="Inference Steps"),
        gr.Number(value=0, label="Seed (0 = random)")
    ],
    outputs=gr.Image(label="Generated Character"),
    title="🎮 AI Game Character Designer",
    description="Generate game characters using Stable Diffusion"
)

interface.launch(share=True)